# 章节介绍

就是增强检索生成技术,其全称是 Retrieval Augmented Generation
本章我们只学习RAG的基本流程与部分原理,目的是了解RAG,知道他在项目中的作用

# LLM的局限

首先，是幻觉问题（hallucination），因为 llm 本身是基于从大量数据中训练出来的概率模型来一个个生成 token，也就是他并没有逻辑和事实基线，所以我们说 llm 的智能是涌现性的智能，是基于概率产生的“伪智能”，而不是底层基于逻辑和推理能力“真智能”。但这对应用层无所谓，因为这个涌现性的只能已经足够强大，在应用层已经表现出了足够使用的逻辑和推理能力。

既说 “不是底层基于逻辑和推理能力”，又说“足够使用的逻辑和推理能力”，那到底有没有智能，什么是涌现性的只能。我们用一个经典的例子说明，我们给猴子一个打字机，让他随便打字，如果这个实验拉长到时间是无限的，是不是总有一天，他会打出一部完整的莎士比亚的小说？ 那是肯定的，因为时间是无限的，他是随机打字，那就一定会在某个时间点所有概率都碰巧了，成了一本莎士比亚的小说。

那一个问题来了，猴子到底懂不懂莎士比亚？那肯定是完全不懂的，它不具备逻辑，只是一切概率性的巧合凑到一起了罢了。而 llm 你可以理解成为一个更大概率打出莎士比亚的猴子，它不理解输出文本的逻辑，更不理解内容背后的逻辑，但因为它的模型足够大，训练数据集足够大，他输出正确内容的概率也足够大。所以，从外界看来，他就像真正理解内容一样，也就像具有真正的逻辑和推理能力。

那我们聊目前 llm 的第二个问题，对领域知识的欠缺。这部分分两种情况，第一种是对知识的更新慢，例如你问他最新的新闻他肯定是不知道的，因为他的训练数据集不可能每天更新；第二种是特定领域的知识不了解，例如你要创建一个宠物医疗 chat bot，他本身训练数据集这方面的知识占比肯定是少的，就很容易出现幻想问题，然后瞎回答。更不要说公司内部的知识库了。

所以，RAG 就是针对这两点进行解决的。

# Rag的原理

RAG就是去增强大模型正确输出的可能性, Rag的基本流程如下:
1. 用户输入提问
2. 检索：根据用户提问对 向量数据库 进行相似性检测，查找与回答用户问题最相关的内容
3. 增强：根据检索的结果，生成 prompt。 一般都会涉及 “仅依赖下述信息源来回答问题” 这种限制 llm 参考信息源的语句，来减少幻想，让回答更加聚焦
4. 生成：将增强后的 prompt 传递给 llm，返回数据给用户

所以 RAG 就是哪里有问题解决哪里，既然大模型无法获得最新和内部的数据集，那我们就使用外挂的向量数据库为 llm 提供最新和内部的数据库。既然大模型有幻想问题，我们就将回答问题所需要的信息和知识编码到上下文中，强制大模型只参考这些内容进行回答。

RAG 更底层的逻辑是，也是我们对待 llm 正确的态度：llm 是逻辑推理引擎，而不是信息引擎。所以，由外挂的向量数据库提供最有效的知识，然后由 llm 根据知识进行推理，提供有价值的回复


# RAG流程

我们现在快速宏观的看看如果打造一个RAG chat bot

## 1.加载数据

因为想要根据用户的提问进行语义检索,我们需要将数据集放到向量数据库中,所以我们需要将不用的数据源加载进来. 这里就设计到了多种数据源,比如pdf code等等
这里 langchain 提供非常丰富的集成工具，帮助我们加载来自多个数据源的数据。这个详细的内容会在对应章节介绍场景的数据源的加载方式

## 2.切分数据

因为大模型的上下文窗口是有限的,而我们很多数据源都很容易比这个大. 更何况，用户的提问经常涉及多个数据源，所以我们需要对数据集进行语意化的切分，根据内容的特点和目标大模型的特点、上下文窗口等，对数据源进行合适的切分。

这里听起来比较容易，但考虑到数据源的多种多样和自然语言的特点，事实上切分函数的选择和参数的设定是非常难以控制的。理论上我们是希望每个文档块都是语意相关，并且相互独立的。

## 3.嵌入

这部分对没有机器学习相关背景的同学不容易理解。这里我们用最简单的词袋（words bag）模型来描述一下最简单的 `embedding(嵌入)` 过程，让大家更具象化的理解这个。

1. 词袋模型就是最简化的情况，把一篇 句子/文章 中的单词提前出来，就像放到一个袋子里一样，认为单词之间是独立的，并不关心词与词之间的上下文关系。
2. 假设我们有十篇英语文章，那我们可以把每个文章拆分成单词，并且还原成最初的形势（例如 did、does => do），然后我们统计每个词出现的次数。 我们简化一下假设最后结果就是
```js
第一篇文章: 
apple: 10, phone:12

第二篇文章:
apple: 8, android: 10, phone: 18

第三篇文章:
banana: 6, juice: 10
```
3. 那我们尝试构建一个向量，也就是一个数组，每个位置有一个值，代表每个单词在这个文章中出现的次数
```js
变量
[apple, banana, phone, android, juice]
```
那每篇文章，都能用一个变量来表示
```js
第一篇文章: [10, 0, 12, 0, 0]
第二篇文章: [8, 0, 18, 10, 0]
第三篇文章: [0, 6, 0, 0, 10]
```
4. 这样我们就能把一篇文章用一个向量来表示了，然后我们可以用最简单的余弦定理去计算两个向量之间的夹角，以此确定两个向量的距离。 
5. 这样，我们就有了通过向量和向量之间的余弦夹角的，来衡量文章之间相似度的能力，是不是很简单。

当然，这是最最最简单的 embedding 原理，不过是所有的 embedding 和相似性搜索都是类似的原理。
回到我们 RAG 流程中，我们将切分后的每一个文档块使用 embedding 算法转换成一个向量，存储到向量数据库中（vector store）中。这样，每一个原始数据都有一个对应的向量，可以用来检索。

## 4. 检索数据

当所有需要的数据都存储到向量数据库中后，我们就可以把用户的提问也 embedding 成向量，用这个向量去向量数据库中进行检索，找到相似性最高的几个文档块，返回。

在这里，使用什么算法去计算向量之间的距离也是需要选择的，这个我们会在对应的章节进行介绍。

## 5.增强 prompt

在有了跟用户提问最相关的文档块后，我们根据文档块去构建 prompt。 一般格式都类似于
```js

你是一个 xxx 的聊天机器人，你的任务是根据给定的文档回答用户问题，并且回答时仅根据给定的文档，尽可能回答
用户问题。如果你不知道，你可以回答“我不知道”。

这是文档:
{docs}

用户的提问是:
{question}
```

## 6.生成

然后就是将组装好的 prompt 传递给 chatbot 进行生成回答。